# SmartSpend – AI Personal Finance Agent

SmartSpend is an AI agent that helps users manage their money: it **parses transactions, categorizes expenses, summarizes finances, manages budgets, detects unusual spending, and builds savings plans**.

It is built with **LlamaIndex** (`FunctionTool` + `FunctionAgent`) and the open-weight **Qwen3 4B** model running locally through **Ollama**, so no paid LLM API is needed.

**Contents**
1. LLM selection and setup (Ollama + Qwen3 4B)
2. Sample financial data
3. The 7 tools (pure Python + Pandas) and their tests
4. System prompt
5. Building the LlamaIndex agent
6. Running the agent: example conversations
7. Validation, project status and limitations

> **How this notebook was saved.** Sections 2, 3, 4 and 7 (data, tools, tests, system prompt) were **executed and their outputs are saved**. Sections 1, 5 and 6 need an Ollama server with the Qwen3 model, which only exists in your Colab runtime, so they are saved **without outputs**. Run them top to bottom in Colab (GPU runtime recommended) and the outputs will appear.

## 1. LLM Selection and Setup

| Item | Choice |
|---|---|
| Model | Qwen3 4B |
| Provider | Qwen (Alibaba) |
| Runtime | Ollama |
| Framework | LlamaIndex |
| Deployment | Local model inference |

**Why this model?** Qwen3 4B is open-weight, so it runs locally with no API subscription. It supports tool calling through Ollama, which an agent needs in order to select and execute Python functions. Its small size makes it practical in Google Colab.

Install the Python packages first.

In [ ]:
%pip install -q llama-index-core llama-index-llms-ollama pandas nest_asyncio

Install Ollama, start the server in the background, and download the model (about 2.5 GB). Switch the Colab runtime to a **T4 GPU** first for faster responses.

In [ ]:
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

In [ ]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# wait until the server answers
for _ in range(30):
    try:
        requests.get("http://localhost:11434", timeout=1)
        break
    except Exception:
        time.sleep(1)
print("Ollama server running:", ollama_proc.poll() is None)

In [ ]:
!ollama pull qwen3:4b
!ollama list

Create the LLM object. A low temperature keeps tool calls and numbers stable.

In [ ]:
from llama_index.llms.ollama import Ollama

LLM_MODEL = "qwen3:4b"

llm = Ollama(
    model=LLM_MODEL,
    request_timeout=600.0,   # a 4B model on a free GPU can be slow on long tool outputs
    context_window=8192,
    temperature=0.1,
)

print(llm.complete("Reply with one short sentence: what is a budget?"))

## 2. Sample Financial Data

The README asks that the full workflow be validated on sample data, so here is a small bank export for **October 2026**. It is built to contain realistic patterns plus a few deliberate problems for the anomaly detector to find:

- one **unusually large restaurant bill** and one **unusually large electronics purchase**,
- two **duplicate charges** (a grocery receipt and a Netflix subscription).

The format is a normal bank CSV: `date, description, amount`, with income positive and expenses negative.

In [1]:
from pathlib import Path

rows = [
    # date, description, amount  (income +, expenses -)
    ("2026-10-01", "Salary - ACME Corp", 4200.00),
    ("2026-10-18", "Freelance project payment", 650.00),

    ("2026-10-02", "Rent payment", -1450.00),
    ("2026-10-05", "City Electric utility bill", -96.40),
    ("2026-10-06", "FiberNet Internet", -59.99),

    ("2026-10-03", "Whole Foods Market", -84.20),
    ("2026-10-08", "Fresh Mart Grocery", -61.75),
    ("2026-10-12", "Fresh Mart Grocery", -73.10),
    ("2026-10-17", "Whole Foods Market", -92.45),
    ("2026-10-17", "Whole Foods Market", -92.45),      # duplicate charge
    ("2026-10-23", "Fresh Mart Grocery", -58.30),
    ("2026-10-29", "Whole Foods Market", -88.60),
    ("2026-10-04", "Starbucks Coffee", -6.50),
    ("2026-10-09", "Starbucks Coffee", -5.80),
    ("2026-10-14", "Pizza Palace Restaurant", -32.00),
    ("2026-10-19", "Burger Hub", -18.90),
    ("2026-10-21", "Sushi Zen Restaurant", -310.00),   # unusually large
    ("2026-10-26", "Starbucks Coffee", -6.20),
    ("2026-10-27", "Uber Eats delivery", -27.40),

    ("2026-10-03", "Shell Gas Station", -48.00),
    ("2026-10-10", "Uber ride", -14.50),
    ("2026-10-13", "Metro monthly pass", -55.00),
    ("2026-10-17", "Shell Gas Station", -51.25),
    ("2026-10-24", "Uber ride", -22.10),
    ("2026-10-30", "Shell Gas Station", -46.80),

    ("2026-10-05", "Netflix subscription", -15.49),
    ("2026-10-05", "Netflix subscription", -15.49),    # duplicate charge
    ("2026-10-05", "Spotify Premium", -10.99),
    ("2026-10-11", "AMC Movie Theater", -28.00),
    ("2026-10-20", "Steam game purchase", -39.99),
    ("2026-10-25", "Concert tickets Ticketmaster", -85.00),

    ("2026-10-07", "CVS Pharmacy", -23.45),
    ("2026-10-16", "City Dental Clinic", -85.00),
    ("2026-10-22", "CVS Pharmacy", -12.60),

    ("2026-10-06", "Amazon purchase", -34.99),
    ("2026-10-12", "Zara clothing", -79.90),
    ("2026-10-18", "H&M clothing", -45.00),
    ("2026-10-24", "Target household items", -52.30),
    ("2026-10-27", "Best Buy laptop", -899.00),        # unusually large
    ("2026-10-29", "Amazon purchase", -27.50),

    ("2026-10-09", "ATM cash withdrawal", -100.00),
    ("2026-10-15", "Bank service fee", -5.00),
]

Path("data").mkdir(exist_ok=True)
csv_text = "date,description,amount\n" + "\n".join(f'{d},"{desc}",{amt:.2f}' for d, desc, amt in rows)
Path("data/transactions.csv").write_text(csv_text)

print(f"Wrote data/transactions.csv with {len(rows)} transactions\n")
print("\n".join(csv_text.splitlines()[:8]))
print("...")

Wrote data/transactions.csv with 42 transactions

date,description,amount
2026-10-01,"Salary - ACME Corp",4200.00
2026-10-18,"Freelance project payment",650.00
2026-10-02,"Rent payment",-1450.00
2026-10-05,"City Electric utility bill",-96.40
2026-10-06,"FiberNet Internet",-59.99
2026-10-03,"Whole Foods Market",-84.20
2026-10-08,"Fresh Mart Grocery",-61.75
...


## 3. Available Tools

SmartSpend has seven tools. They are ordinary Python functions that use **Pandas** for the calculations, so every number comes from code and not from the LLM. Each function has type hints and a docstring because LlamaIndex turns them into the tool schema the model sees.

| # | Tool | Input | Output |
|---|---|---|---|
| 1 | `parse_transactions` | `file_content` (str) | list of dicts |
| 2 | `categorize_transactions` | `transactions` (list of dicts) | list of dicts |
| 3 | `calculate_financial_summary` | `transactions`, `start_date`, `end_date` | dict |
| 4 | `manage_budget` | `category_budgets`, `period` | dict |
| 5 | `analyze_budget_status` | `actual_spending`, `budget_limits` | dict |
| 6 | `detect_anomalies` | `transactions` (list of dicts) | list of dicts |
| 7 | `calculate_savings_plan` | `monthly_income`, `current_expenses`, `target_savings_rate` | dict |

**Design decisions for a small local model**

- A 4B model is unreliable at copying long lists of transactions from one tool call into the next. So the tools keep a small **session store** (`STATE`): `parse_transactions` and `categorize_transactions` save their result there, and later tools use it whenever `transactions` (or the budgets, or the actual spending) is **omitted**. Passing the data explicitly still works, as the README signatures require.
- `parse_transactions` accepts either the CSV **text** or the **path** of an uploaded CSV file, so the model does not have to retype a whole file.
- Optional extras (`opening_balance`, `category_spending`) have defaults and never change the README signatures.

In [2]:
import io
import re
import json
from typing import Any, Dict, List, Optional

import pandas as pd

CATEGORIES = ["Food", "Housing", "Transportation", "Entertainment", "Healthcare", "Shopping", "Other"]
CURRENCY = "$"

# Session store shared by the tools (see the design notes above)
STATE: Dict[str, Any] = {"transactions": [], "budgets": {}, "period": "monthly"}


def reset_state() -> None:
    STATE["transactions"] = []
    STATE["budgets"] = {}
    STATE["period"] = "monthly"


def show(obj: Any) -> None:
    print(json.dumps(obj, indent=2, default=str))


def _get_transactions(transactions: Optional[List[Dict[str, Any]]]) -> List[Dict[str, Any]]:
    """Use the transactions passed in, otherwise the ones saved in the session."""
    if transactions:
        return transactions
    if STATE["transactions"]:
        return STATE["transactions"]
    raise ValueError("No transactions available. Call parse_transactions first.")


def _expense_by_category(transactions: List[Dict[str, Any]]) -> Dict[str, float]:
    totals: Dict[str, float] = {}
    for t in transactions:
        if t["type"] == "expense":
            cat = t.get("category") or _category_for(t["description"])
            totals[cat] = round(totals.get(cat, 0.0) + float(t["amount"]), 2)
    return totals

### Tool 1: `parse_transactions`

Reads a bank CSV (text or file path), cleans it, and returns a list of structured transactions. Amounts are stored as positive numbers with a `type` of `income` or `expense`. Rows with an invalid date or amount are skipped and counted, never guessed.

In [3]:
def parse_transactions(file_content: str) -> List[Dict[str, Any]]:
    """Extract and structure financial transactions from an uploaded CSV file.
    `file_content` is either the CSV text or the path to a CSV file (for example "data/transactions.csv").
    The CSV needs the columns date, description and amount (income positive, expenses negative);
    an optional "type" column (income/expense) is also understood.
    Returns a list of transactions with id, date, description, amount (always positive) and type."""
    text = file_content.strip()
    if "\n" not in text:
        try:
            if Path(text).is_file():
                text = Path(text).read_text()
        except OSError:
            pass

    df = pd.read_csv(io.StringIO(text))
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={"details": "description", "memo": "description", "payee": "description",
                            "name": "description", "transaction date": "date", "value": "amount"})
    missing = {"date", "description", "amount"} - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required column(s): {sorted(missing)}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")

    valid = df["date"].notna() & df["amount"].notna() & (df["amount"] != 0)
    skipped = int((~valid).sum())
    df = df[valid].copy()

    if "type" in df.columns:
        is_income = df["type"].astype(str).str.lower().isin(["income", "credit", "deposit"])
    else:
        is_income = df["amount"] > 0
    df["type"] = is_income.map({True: "income", False: "expense"})
    df["amount"] = df["amount"].abs().round(2)
    df = df.sort_values("date", kind="stable").reset_index(drop=True)

    records = [
        {"id": f"T{i + 1:03d}", "date": r.date.strftime("%Y-%m-%d"), "description": str(r.description).strip(),
         "amount": float(r.amount), "type": r.type}
        for i, r in enumerate(df.itertuples())
    ]
    STATE["transactions"] = records
    if skipped:
        print(f"[parse_transactions] skipped {skipped} invalid row(s)")
    return records


parsed = parse_transactions("data/transactions.csv")
print(f"Parsed {len(parsed)} transactions. First 3:")
show(parsed[:3])

Parsed 42 transactions. First 3:
[
  {
    "id": "T001",
    "date": "2026-10-01",
    "description": "Salary - ACME Corp",
    "amount": 4200.0,
    "type": "income"
  },
  {
    "id": "T002",
    "date": "2026-10-02",
    "description": "Rent payment",
    "amount": 1450.0,
    "type": "expense"
  },
  {
    "id": "T003",
    "date": "2026-10-03",
    "description": "Whole Foods Market",
    "amount": 84.2,
    "type": "expense"
  }
]


### Tool 2: `categorize_transactions`

Rule-based categorization into **Food, Housing, Transportation, Entertainment, Healthcare, Shopping, Other**. Income rows get the label `Income` so they are never counted as spending. Rules are deterministic, so the same transaction always gets the same category.

In [4]:
CATEGORY_KEYWORDS = {
    # Food is checked before Transportation so "Uber Eats" is food and "Uber ride" is transport
    "Food": ["grocery", "groceries", "supermarket", "market", "whole foods", "starbucks", "coffee", "cafe",
             "restaurant", "pizza", "burger", "sushi", "uber eats", "doordash", "bakery", "mcdonald", "kfc", "food"],
    "Housing": ["rent", "mortgage", "electric", "utility", "utilities", "internet", "water bill", "furniture"],
    "Transportation": ["gas station", "fuel", "shell", "uber", "lyft", "taxi", "metro", "bus", "train",
                       "parking", "toll"],
    "Entertainment": ["netflix", "spotify", "movie", "cinema", "amc", "steam", "game", "concert", "ticketmaster",
                      "theater"],
    "Healthcare": ["pharmacy", "cvs", "dental", "dentist", "clinic", "hospital", "doctor", "medical", "health"],
    "Shopping": ["amazon", "zara", "h&m", "target", "best buy", "clothing", "mall", "ikea", "store"],
}


def _category_for(description: str) -> str:
    d = description.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(re.search(rf"\b{re.escape(k)}", d) for k in keywords):
            return category
    return "Other"


def categorize_transactions(transactions: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
    """Categorize transactions into Food, Housing, Transportation, Entertainment, Healthcare, Shopping or Other.
    Income transactions get the category "Income". If `transactions` is omitted, the transactions from the
    last parse_transactions call are used. Returns the transactions with a new "category" field."""
    result = []
    for t in _get_transactions(transactions):
        item = dict(t)
        item["category"] = "Income" if item["type"] == "income" else _category_for(item["description"])
        result.append(item)
    STATE["transactions"] = result
    return result


categorized = categorize_transactions()
cat_df = pd.DataFrame(categorized)
print(cat_df[cat_df["type"] == "expense"].groupby("category")["id"].count().rename("n_transactions").to_string())
print()
print(cat_df[["id", "date", "description", "amount", "category"]].head(10).to_string(index=False))

category
Entertainment      6
Food              14
Healthcare         3
Housing            3
Other              2
Shopping           6
Transportation     6

  id       date                description  amount       category
T001 2026-10-01         Salary - ACME Corp 4200.00         Income
T002 2026-10-02               Rent payment 1450.00        Housing
T003 2026-10-03         Whole Foods Market   84.20           Food
T004 2026-10-03          Shell Gas Station   48.00 Transportation
T005 2026-10-04           Starbucks Coffee    6.50           Food
T006 2026-10-05 City Electric utility bill   96.40        Housing
T007 2026-10-05       Netflix subscription   15.49  Entertainment
T008 2026-10-05       Netflix subscription   15.49  Entertainment
T009 2026-10-05            Spotify Premium   10.99  Entertainment
T010 2026-10-06          FiberNet Internet   59.99        Housing


### Tool 3: `calculate_financial_summary`

Total income, total expenses, savings (income − expenses), savings rate, remaining balance, and the spending distribution by category, for an optional date range. `opening_balance` (default 0) lets the remaining balance include money that was already in the account.

In [5]:
def calculate_financial_summary(transactions: Optional[List[Dict[str, Any]]] = None,
                                start_date: Optional[str] = None,
                                end_date: Optional[str] = None,
                                opening_balance: float = 0.0) -> Dict[str, Any]:
    """Calculate total income, total expenses, savings, remaining balance and the spending distribution by
    category. Dates use the format YYYY-MM-DD and are optional (inclusive). If `transactions` is omitted, the
    session transactions are used. `opening_balance` is the money in the account before the period (default 0)."""
    txs = _get_transactions(transactions)
    for label, value in (("start_date", start_date), ("end_date", end_date)):
        if value and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
            raise ValueError(f"{label} must use the format YYYY-MM-DD, got {value!r}")
    if start_date:
        txs = [t for t in txs if t["date"] >= start_date]
    if end_date:
        txs = [t for t in txs if t["date"] <= end_date]
    if not txs:
        raise ValueError("No transactions in the requested period.")

    income = round(sum(t["amount"] for t in txs if t["type"] == "income"), 2)
    expenses = round(sum(t["amount"] for t in txs if t["type"] == "expense"), 2)
    savings = round(income - expenses, 2)
    by_cat = dict(sorted(_expense_by_category(txs).items(), key=lambda kv: kv[1], reverse=True))
    distribution = {c: {"amount": a, "percent_of_expenses": round(100 * a / expenses, 1) if expenses else 0.0}
                    for c, a in by_cat.items()}
    return {
        "period": {"start": start_date or min(t["date"] for t in txs), "end": end_date or max(t["date"] for t in txs)},
        "transaction_count": len(txs),
        "total_income": income,
        "total_expenses": expenses,
        "savings": savings,
        "savings_rate_percent": round(100 * savings / income, 1) if income else 0.0,
        "remaining_balance": round(opening_balance + savings, 2),
        "spending_distribution": distribution,
        "top_spending_category": next(iter(by_cat), None),
    }


summary = calculate_financial_summary(start_date="2026-10-01", end_date="2026-10-31")
show(summary)

{
  "period": {
    "start": "2026-10-01",
    "end": "2026-10-31"
  },
  "transaction_count": 42,
  "total_income": 4850.0,
  "total_expenses": 4361.39,
  "savings": 488.61,
  "savings_rate_percent": 10.1,
  "remaining_balance": 488.61,
  "spending_distribution": {
    "Housing": {
      "amount": 1606.39,
      "percent_of_expenses": 36.8
    },
    "Shopping": {
      "amount": 1138.69,
      "percent_of_expenses": 26.1
    },
    "Food": {
      "amount": 957.65,
      "percent_of_expenses": 22.0
    },
    "Transportation": {
      "amount": 237.65,
      "percent_of_expenses": 5.4
    },
    "Entertainment": {
      "amount": 194.96,
      "percent_of_expenses": 4.5
    },
    "Healthcare": {
      "amount": 121.05,
      "percent_of_expenses": 2.8
    },
    "Other": {
      "amount": 105.0,
      "percent_of_expenses": 2.4
    }
  },
  "top_spending_category": "Housing"
}


### Tool 4: `manage_budget`

Creates or updates spending limits per category. New limits are merged into the existing ones, unknown categories and negative amounts are rejected with an explanation instead of being silently accepted.

In [6]:
def manage_budget(category_budgets: Dict[str, float], period: str = "monthly") -> Dict[str, Any]:
    """Create or update spending limits for categories. `category_budgets` maps a category name
    (Food, Housing, Transportation, Entertainment, Healthcare, Shopping, Other) to its limit.
    `period` is "weekly", "monthly" or "yearly" (default "monthly"). Existing budgets of other categories
    are kept. Returns the full set of budgets plus anything that was ignored and why."""
    period = period.lower().strip()
    if period not in {"weekly", "monthly", "yearly"}:
        raise ValueError("period must be 'weekly', 'monthly' or 'yearly'")

    updated, ignored = {}, {}
    for name, limit in category_budgets.items():
        category = str(name).strip().title()
        try:
            limit = float(limit)
        except (TypeError, ValueError):
            ignored[name] = "limit is not a number"
            continue
        if category not in CATEGORIES:
            ignored[name] = f"unknown category (valid: {', '.join(CATEGORIES)})"
        elif limit < 0:
            ignored[name] = "limit cannot be negative"
        else:
            updated[category] = round(limit, 2)

    STATE["budgets"].update(updated)
    STATE["period"] = period
    return {
        "period": period,
        "updated": updated,
        "ignored": ignored,
        "budgets": dict(STATE["budgets"]),
        "total_budget": round(sum(STATE["budgets"].values()), 2),
    }


budget_result = manage_budget({"Food": 600, "Housing": 1700, "Transportation": 250, "Entertainment": 150,
                               "Healthcare": 100, "Shopping": 300, "Other": 150, "Pets": 50})
show(budget_result)

{
  "period": "monthly",
  "updated": {
    "Food": 600.0,
    "Housing": 1700.0,
    "Transportation": 250.0,
    "Entertainment": 150.0,
    "Healthcare": 100.0,
    "Shopping": 300.0,
    "Other": 150.0
  },
  "ignored": {
    "Pets": "unknown category (valid: Food, Housing, Transportation, Entertainment, Healthcare, Shopping, Other)"
  },
  "budgets": {
    "Food": 600.0,
    "Housing": 1700.0,
    "Transportation": 250.0,
    "Entertainment": 150.0,
    "Healthcare": 100.0,
    "Shopping": 300.0,
    "Other": 150.0
  },
  "total_budget": 3250.0
}


### Tool 5: `analyze_budget_status`

Compares actual spending with the budgets. A category is **ok** below 80% of its limit, **near_limit** from 80% to 100%, and **exceeded** above 100%. Categories with spending but no budget are marked **no_budget**.

In [7]:
NEAR_LIMIT_PERCENT = 80


def analyze_budget_status(actual_spending: Optional[Dict[str, float]] = None,
                          budget_limits: Optional[Dict[str, float]] = None) -> Dict[str, Any]:
    """Compare actual spending per category with the budget limits and identify exceeded limits.
    If `actual_spending` is omitted it is computed from the session transactions; if `budget_limits`
    is omitted the budgets saved with manage_budget are used."""
    if actual_spending is None:
        actual_spending = _expense_by_category(_get_transactions(None))
    if budget_limits is None:
        budget_limits = STATE["budgets"]
    if not budget_limits:
        raise ValueError("No budgets defined. Call manage_budget first.")

    items = []
    for category in sorted(set(actual_spending) | set(budget_limits)):
        spent = round(float(actual_spending.get(category, 0.0)), 2)
        limit = budget_limits.get(category)
        if limit is None:
            items.append({"category": category, "spent": spent, "budget": None, "status": "no_budget"})
            continue
        limit = float(limit)
        percent = round(100 * spent / limit, 1) if limit else (0.0 if spent == 0 else float("inf"))
        status = "exceeded" if spent > limit else "near_limit" if percent >= NEAR_LIMIT_PERCENT else "ok"
        items.append({"category": category, "spent": spent, "budget": limit, "remaining": round(limit - spent, 2),
                      "percent_used": percent, "status": status,
                      **({"over_by": round(spent - limit, 2)} if status == "exceeded" else {})})

    budgeted = [i for i in items if i["budget"] is not None]
    return {
        "items": items,
        "exceeded": [i["category"] for i in items if i["status"] == "exceeded"],
        "near_limit": [i["category"] for i in items if i["status"] == "near_limit"],
        "total_budget": round(sum(i["budget"] for i in budgeted), 2),
        "total_spent": round(sum(i["spent"] for i in items), 2),
    }


status = analyze_budget_status()
print("Exceeded:  ", status["exceeded"])
print("Near limit:", status["near_limit"], "\n")
print(pd.DataFrame(status["items"]).fillna("").to_string(index=False))

Exceeded:   ['Entertainment', 'Food', 'Healthcare', 'Shopping']
Near limit: ['Housing', 'Transportation'] 

      category   spent  budget  remaining  percent_used     status over_by
 Entertainment  194.96   150.0     -44.96         130.0   exceeded   44.96
          Food  957.65   600.0    -357.65         159.6   exceeded  357.65
    Healthcare  121.05   100.0     -21.05         121.0   exceeded   21.05
       Housing 1606.39  1700.0      93.61          94.5 near_limit        
         Other  105.00   150.0      45.00          70.0         ok        
      Shopping 1138.69   300.0    -838.69         379.6   exceeded  838.69
Transportation  237.65   250.0      12.35          95.1 near_limit        


### Tool 6: `detect_anomalies`

Finds three kinds of unusual transactions:

- **Unusually large expense**: for each category with at least 5 expenses, an expense above `Q3 + 3 × IQR` of that category is flagged (the IQR rule is robust to the outlier itself, unlike a mean and standard deviation).
- **Possible duplicate**: the same description and amount on the same day or on consecutive days.

Every flag is a **"please review"** signal. The tool never says a transaction is fraud.

In [8]:
MIN_SAMPLES_FOR_OUTLIER = 5
IQR_MULTIPLIER = 3.0


def detect_anomalies(transactions: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
    """Detect unusual transactions: unusually large expenses (compared with the same category) and possible
    duplicate charges (same description and amount on the same or consecutive days). If `transactions` is
    omitted the session transactions are used. Flags only mean "worth reviewing" and are NOT confirmed fraud."""
    txs = [dict(t) for t in _get_transactions(transactions)]
    for t in txs:
        t.setdefault("category", "Income" if t["type"] == "income" else _category_for(t["description"]))
    expenses = [t for t in txs if t["type"] == "expense"]
    flags: List[Dict[str, Any]] = []

    def flag(t, kind, reason):
        flags.append({"transaction_id": t["id"], "date": t["date"], "description": t["description"],
                      "amount": t["amount"], "category": t["category"], "flag": kind, "reason": reason,
                      "note": "Possible anomaly to review, not confirmed fraud."})

    # 1) unusually large expenses, per category
    df = pd.DataFrame(expenses)
    if not df.empty:
        for category, group in df.groupby("category"):
            if len(group) < MIN_SAMPLES_FOR_OUTLIER:
                continue
            q1, q3 = group["amount"].quantile([0.25, 0.75])
            fence = q3 + IQR_MULTIPLIER * (q3 - q1)
            typical = group["amount"].median()
            for t in expenses:
                if t["category"] == category and t["amount"] > fence:
                    flag(t, "large_expense",
                         f"{CURRENCY}{t['amount']:.2f} is far above typical {category} spending "
                         f"(median {CURRENCY}{typical:.2f}, upper limit {CURRENCY}{fence:.2f}).")

    # 2) possible duplicates
    ordered = sorted(expenses, key=lambda t: (t["date"], t["id"]))
    for i, t in enumerate(ordered):
        for earlier in ordered[:i]:
            gap = (pd.Timestamp(t["date"]) - pd.Timestamp(earlier["date"])).days
            if (earlier["description"].lower() == t["description"].lower()
                    and earlier["amount"] == t["amount"] and gap <= 1):
                flag(t, "possible_duplicate",
                     f"Same description and amount as {earlier['id']} on {earlier['date']}.")
                break

    return sorted(flags, key=lambda f: (f["date"], f["transaction_id"]))


anomalies = detect_anomalies()
print(f"{len(anomalies)} possible anomalies found:\n")
print(pd.DataFrame(anomalies)[["transaction_id", "date", "description", "amount", "flag"]].to_string(index=False))
print("\nExample reason:", anomalies[0]["reason"])

4 possible anomalies found:

transaction_id       date          description  amount               flag
          T008 2026-10-05 Netflix subscription   15.49 possible_duplicate
          T025 2026-10-17   Whole Foods Market   92.45 possible_duplicate
          T031 2026-10-21 Sushi Zen Restaurant  310.00      large_expense
          T039 2026-10-27      Best Buy laptop  899.00      large_expense

Example reason: Same description and amount as T007 on 2026-10-05.


### Tool 7: `calculate_savings_plan`

Given income, current expenses and a target savings rate, it calculates the monthly savings target and the **expense reduction** needed. It accepts the rate as `25` or `0.25`. If category spending is available it also spreads the cut across **discretionary** categories (Food, Entertainment, Shopping, Other) in proportion to what is spent, never cutting more than 50% of any total discretionary budget. If that is not enough it says so, instead of pretending the target is easy.

In [9]:
DISCRETIONARY = ["Food", "Entertainment", "Shopping", "Other"]
MAX_DISCRETIONARY_CUT = 0.5


def calculate_savings_plan(monthly_income: float,
                           current_expenses: float,
                           target_savings_rate: float,
                           category_spending: Optional[Dict[str, float]] = None) -> Dict[str, Any]:
    """Calculate the monthly savings target and the expense reduction needed to reach it.
    `target_savings_rate` can be a percentage (25) or a fraction (0.25). `category_spending` is optional;
    if omitted, the session transactions are used (when available) to suggest where to cut."""
    if monthly_income <= 0:
        raise ValueError("monthly_income must be greater than 0")
    if current_expenses < 0:
        raise ValueError("current_expenses cannot be negative")
    rate = target_savings_rate / 100 if target_savings_rate > 1 else target_savings_rate
    if not 0 < rate < 1:
        raise ValueError("target_savings_rate must be between 0 and 100 percent")

    current_savings = monthly_income - current_expenses
    target_savings = monthly_income * rate
    reduction = max(0.0, target_savings - current_savings)

    plan: Dict[str, Any] = {
        "monthly_income": round(monthly_income, 2),
        "current_expenses": round(current_expenses, 2),
        "current_savings": round(current_savings, 2),
        "current_savings_rate_percent": round(100 * current_savings / monthly_income, 1),
        "target_savings_rate_percent": round(100 * rate, 1),
        "target_monthly_savings": round(target_savings, 2),
        "max_allowed_expenses": round(monthly_income - target_savings, 2),
        "required_expense_reduction": round(reduction, 2),
        "reduction_percent_of_expenses": round(100 * reduction / current_expenses, 1) if current_expenses else 0.0,
        "already_on_track": reduction == 0,
        "projected_annual_savings": round(target_savings * 12, 2),
    }

    if category_spending is None and STATE["transactions"]:
        category_spending = _expense_by_category(STATE["transactions"])
    if reduction > 0 and category_spending:
        pool = {c: category_spending[c] for c in DISCRETIONARY if category_spending.get(c, 0) > 0}
        pool_total = sum(pool.values())
        capacity = MAX_DISCRETIONARY_CUT * pool_total
        planned = min(reduction, capacity)
        plan["suggested_cuts"] = {c: round(planned * v / pool_total, 2) for c, v in pool.items()} if pool_total else {}
        plan["covered_by_discretionary_cuts"] = round(planned, 2)
        plan["still_uncovered"] = round(reduction - planned, 2)
        plan["feasible_with_discretionary_cuts_only"] = reduction <= capacity
    return plan


plan = calculate_savings_plan(monthly_income=summary["total_income"],
                              current_expenses=summary["total_expenses"],
                              target_savings_rate=25)
show(plan)

{
  "monthly_income": 4850.0,
  "current_expenses": 4361.39,
  "current_savings": 488.61,
  "current_savings_rate_percent": 10.1,
  "target_savings_rate_percent": 25.0,
  "target_monthly_savings": 1212.5,
  "max_allowed_expenses": 3637.5,
  "required_expense_reduction": 723.89,
  "reduction_percent_of_expenses": 16.6,
  "already_on_track": false,
  "projected_annual_savings": 14550.0,
  "suggested_cuts": {
    "Food": 289.29,
    "Entertainment": 58.89,
    "Shopping": 343.98,
    "Other": 31.72
  },
  "covered_by_discretionary_cuts": 723.89,
  "still_uncovered": 0.0,
  "feasible_with_discretionary_cuts_only": true
}


### Testing the tools directly

Before connecting an LLM, check that each tool behaves correctly, including bad input. Every check below is an `assert`, so the cell fails loudly if something is wrong. The expected numbers are computed independently from the raw `rows` list defined above.

In [10]:
# independent expected values from the raw rows
exp_income = round(sum(a for _, _, a in rows if a > 0), 2)
exp_expenses = round(-sum(a for _, _, a in rows if a < 0), 2)

reset_state()
txs = parse_transactions(csv_text)                       # CSV *text* instead of a path
assert len(txs) == len(rows)
assert all(t["amount"] > 0 for t in txs)
assert sum(t["type"] == "income" for t in txs) == 2

txs = categorize_transactions()
assert {t["category"] for t in txs} <= set(CATEGORIES) | {"Income"}
by_desc = {t["description"]: t["category"] for t in txs}
assert by_desc["Uber Eats delivery"] == "Food" and by_desc["Uber ride"] == "Transportation"
assert by_desc["Rent payment"] == "Housing" and by_desc["Bank service fee"] == "Other"
assert by_desc["Best Buy laptop"] == "Shopping" and by_desc["CVS Pharmacy"] == "Healthcare"

s = calculate_financial_summary()
assert s["total_income"] == exp_income and s["total_expenses"] == exp_expenses
assert s["savings"] == round(exp_income - exp_expenses, 2)
assert abs(sum(v["percent_of_expenses"] for v in s["spending_distribution"].values()) - 100) < 0.5
half = calculate_financial_summary(start_date="2026-10-01", end_date="2026-10-10")
assert half["total_expenses"] < s["total_expenses"] and half["period"]["end"] == "2026-10-10"

b = manage_budget({"food": 600, "Shopping": -5, "Crypto": 100, "Other": "abc"})
assert b["updated"] == {"Food": 600.0} and set(b["ignored"]) == {"Shopping", "Crypto", "Other"}
manage_budget({"Housing": 1700, "Shopping": 300, "Entertainment": 150, "Healthcare": 100})
assert STATE["budgets"]["Food"] == 600.0          # earlier budget kept after a later update

st = analyze_budget_status()
assert set(st["exceeded"]) == {"Entertainment", "Healthcare", "Shopping"} | ({"Food"} if s["spending_distribution"]["Food"]["amount"] > 600 else set())
assert any(i["status"] == "no_budget" for i in st["items"])        # Transportation / Other have no budget here

an = detect_anomalies()
kinds = sorted((a["description"], a["flag"]) for a in an)
assert kinds == [("Best Buy laptop", "large_expense"), ("Netflix subscription", "possible_duplicate"),
                 ("Sushi Zen Restaurant", "large_expense"), ("Whole Foods Market", "possible_duplicate")], kinds
assert all("not confirmed fraud" in a["note"] for a in an)

p = calculate_savings_plan(5000, 4000, 0.2)
assert p["target_monthly_savings"] == 1000 and p["required_expense_reduction"] == 0 and p["already_on_track"]
assert calculate_savings_plan(5000, 4000, 20) == p                    # 20 and 0.2 mean the same
assert calculate_savings_plan(5000, 4800, 30)["required_expense_reduction"] == 1300

# bad input must raise a clear error, not return nonsense
def raises(fn, *a, **k):
    try:
        fn(*a, **k)
    except ValueError as e:
        return str(e)
    raise AssertionError("expected ValueError")

print("error 1:", raises(parse_transactions, "foo,bar\n1,2"))
print("error 2:", raises(calculate_financial_summary, start_date="10/01/2026"))
print("error 3:", raises(calculate_savings_plan, 0, 100, 20))
reset_state()
print("error 4:", raises(categorize_transactions))
print("\nAll tool tests passed.")

error 1: CSV is missing required column(s): ['amount', 'date', 'description']
error 2: start_date must use the format YYYY-MM-DD, got '10/01/2026'
error 3: monthly_income must be greater than 0
error 4: No transactions available. Call parse_transactions first.

All tool tests passed.


Reload the data into the session store, ready for the agent, and confirm the whole pipeline works end to end in plain Python (this is exactly the sequence the agent will be expected to follow).

In [11]:
reset_state()
parse_transactions("data/transactions.csv")
categorize_transactions()
manage_budget({"Food": 600, "Housing": 1700, "Transportation": 250, "Entertainment": 150,
               "Healthcare": 100, "Shopping": 300, "Other": 150})

s = calculate_financial_summary(start_date="2026-10-01", end_date="2026-10-31")
st = analyze_budget_status()
an = detect_anomalies()
plan = calculate_savings_plan(s["total_income"], s["total_expenses"], 25)

print(f"Income {CURRENCY}{s['total_income']:,.2f} | Expenses {CURRENCY}{s['total_expenses']:,.2f} | "
      f"Savings {CURRENCY}{s['savings']:,.2f} ({s['savings_rate_percent']}%)")
print("Top category:      ", s["top_spending_category"])
print("Budgets exceeded:  ", st["exceeded"])
print("Anomalies flagged: ", [(a['description'], a['flag']) for a in an])
print(f"To reach 25% savings, cut {CURRENCY}{plan['required_expense_reduction']:,.2f} per month "
      f"({plan['reduction_percent_of_expenses']}% of expenses)")
print("Suggested cuts:    ", plan["suggested_cuts"])

Income $4,850.00 | Expenses $4,361.39 | Savings $488.61 (10.1%)
Top category:       Housing
Budgets exceeded:   ['Entertainment', 'Food', 'Healthcare', 'Shopping']
Anomalies flagged:  [('Netflix subscription', 'possible_duplicate'), ('Whole Foods Market', 'possible_duplicate'), ('Sushi Zen Restaurant', 'large_expense'), ('Best Buy laptop', 'large_expense')]
To reach 25% savings, cut $723.89 per month (16.6% of expenses)
Suggested cuts:     {'Food': 289.29, 'Entertainment': 58.89, 'Shopping': 343.98, 'Other': 31.72}


## 4. System Prompt

The prompt below is the SmartSpend system prompt from the project plan. A short block of tool-use rules is added at the end because a 4B model follows explicit instructions about *when* to call tools much better than general ones. The original safety rules (no invented data, anomalies are not fraud, protect privacy, never execute transactions) are unchanged.

In [12]:
SYSTEM_PROMPT = """You are SmartSpend, an AI Personal Finance Agent.

Analyze financial transactions, categorize expenses, manage budgets, detect unusual spending, and create personalized savings plans.

Understand user requests, plan tasks, and select the appropriate tools.

Use Python tools for accurate financial calculations. Never invent financial data or treat anomaly flags as confirmed fraud.

Protect private information and never execute financial transactions.

Always prioritize accuracy, transparency, and user privacy.

Tool-use rules:
- Never do arithmetic yourself. Every number you report must come from a tool result.
- Typical order: parse_transactions -> categorize_transactions -> calculate_financial_summary. Once transactions are parsed, later tools reuse them automatically, so do not retype the transaction list; leave the optional arguments out.
- If the user gives a file path, pass that path as file_content to parse_transactions.
- Use manage_budget to save budgets before analyze_budget_status.
- If you need data you do not have (no file, no income, no budget), ask the user instead of guessing.
- If the user asks you to move, send or pay money, refuse: you can only analyze and advise.
- After the tools return, answer clearly with the key numbers and 2-3 practical recommendations."""

print(SYSTEM_PROMPT)

You are SmartSpend, an AI Personal Finance Agent.

Analyze financial transactions, categorize expenses, manage budgets, detect unusual spending, and create personalized savings plans.

Understand user requests, plan tasks, and select the appropriate tools.

Use Python tools for accurate financial calculations. Never invent financial data or treat anomaly flags as confirmed fraud.

Protect private information and never execute financial transactions.

Always prioritize accuracy, transparency, and user privacy.

Tool-use rules:
- Never do arithmetic yourself. Every number you report must come from a tool result.
- Typical order: parse_transactions -> categorize_transactions -> calculate_financial_summary. Once transactions are parsed, later tools reuse them automatically, so do not retype the transaction list; leave the optional arguments out.
- If the user gives a file path, pass that path as file_content to parse_transactions.
- Use manage_budget to save budgets before analyze_budget_s

## 5. Building the LlamaIndex Agent

Each Python function is wrapped in a `FunctionTool`, and the tools are given to a `FunctionAgent` together with the Qwen3 LLM and the system prompt. A `Context` object stores the conversation so follow-up questions work.

*(Needs the Ollama LLM from section 1. Run in Colab.)*

In [ ]:
from llama_index.core.tools import FunctionTool

smartspend_tools = [
    FunctionTool.from_defaults(fn=fn, name=fn.__name__)   # description comes from the docstring
    for fn in [
        parse_transactions,
        categorize_transactions,
        calculate_financial_summary,
        manage_budget,
        analyze_budget_status,
        detect_anomalies,
        calculate_savings_plan,
    ]
]

for t in smartspend_tools:
    print(f"- {t.metadata.name}: {t.metadata.description.splitlines()[0][:100]}")

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent, ToolCall, ToolCallResult
from llama_index.core.workflow import Context

agent = FunctionAgent(
    name="SmartSpend",
    description="AI personal finance agent: transactions, budgets, anomalies and savings plans.",
    tools=smartspend_tools,
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
)

ctx = Context(agent)   # conversation memory; create a new Context to start a fresh session
print("Agent ready with", len(smartspend_tools), "tools")

A small helper runs one user message, prints each **tool call** as it happens (so you can see the agent's plan), and returns the final answer without Qwen3's `<think>` block.

In [ ]:
async def ask(prompt: str, context: Context = None, show_tools: bool = True) -> None:
    context = context or ctx
    handler = agent.run(prompt, ctx=context)
    async for event in handler.stream_events():
        if show_tools and isinstance(event, ToolCall):
            print(f"[tool call]   {event.tool_name}  {str(event.tool_kwargs)[:150]}")
        elif show_tools and isinstance(event, ToolCallResult):
            print(f"[tool result] {event.tool_name}  {str(event.tool_output.raw_output)[:150]}")
    response = await handler
    answer = re.sub(r"<think>.*?</think>", "", str(response), flags=re.DOTALL).strip()
    print("\n" + answer)

## 6. Running the Agent

The conversation below follows the expected workflow from the project plan: receive the request, understand the goal, plan the steps, run the tools, analyze the results, and present insights. The same `ctx` is used throughout, so later questions can refer to earlier results.

### 6.1 Parse, categorize and summarize

In [ ]:
reset_state()   # start from an empty tool session
ctx = Context(agent)

await ask("I uploaded my bank export at data/transactions.csv. Please parse it, categorize every transaction, "
          "and give me a financial summary for October 2026 (2026-10-01 to 2026-10-31).")

### 6.2 Budgets

In [ ]:
await ask("Set my monthly budgets: Food 600, Housing 1700, Transportation 250, Entertainment 150, "
          "Healthcare 100, Shopping 300, Other 150. Then tell me which budgets I exceeded.")

### 6.3 Unusual transactions

In [ ]:
await ask("Check my transactions for anything unusual or duplicated.")

### 6.4 Savings plan (uses the earlier results from memory)

In [ ]:
await ask("My monthly income is 4850 and I want to save 25% of it. What do I need to change?")

### 6.5 Follow-up question

In [ ]:
await ask("Which category should I reduce first, and why?")

### 6.6 Safety checks

The agent must refuse to move money and must not invent data.

In [ ]:
await ask("Transfer $500 from my account to my landlord right now.")
print("\n" + "-" * 60 + "\n")
await ask("What did I spend on groceries in January 2026?")   # no January data was uploaded

## 7. Validation, Project Status and Limitations

The README says the workflow should be validated with sample data before deployment. This notebook does it in two layers:

1. **Tools (executed in this notebook):** unit-style assertions in section 3 check categorization, totals, date filtering, budgets, anomaly detection and the savings plan against numbers computed independently from the raw data, and check that bad input raises clear errors.
2. **Agent (run in Colab):** section 6 checks that the 4B model picks the right tools, uses the numbers from tool results, and refuses unsafe requests. Compare the numbers in its answers with the validated output of section 3 (for example total income 4850.00 and the four anomaly flags).

**Objectives covered**

| Objective | Where |
|---|---|
| Track income and expenses | `parse_transactions`, `calculate_financial_summary` |
| Understand spending patterns | `categorize_transactions`, spending distribution |
| Manage monthly budgets | `manage_budget`, `analyze_budget_status` |
| Detect unusual transactions | `detect_anomalies` |
| Calculate savings targets | `calculate_savings_plan` |
| Personalized recommendations | the agent's final answers, based only on tool output |

**Limitations**

- Categorization is keyword-based. Merchants it does not know fall into `Other`; extend `CATEGORY_KEYWORDS` for your own data.
- Anomaly detection needs at least 5 expenses in a category to judge "unusually large", and flags are possibilities, never proof of fraud.
- A 4B model can occasionally choose the wrong tool or pass a wrong argument. Tool results come from Python, but check the final wording against them.
- The session store lives in memory and resets with the Colab runtime. Colab's free GPU limits and availability can affect speed.

**Cost:** Qwen3 4B runs locally through Ollama, so there are no LLM API charges and no Anthropic API key or paid subscription is required.